### Imports

In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
import os

### Load ResNet18 Model + GRU

In [ ]:
class Experiment3(nn.Module):
    def __init__(self, num_classes=4):
        super(Experiment3, self).__init__()
        # load pretrained model
        resnet_model = resnet18(weights=ResNet18_Weights.DEFAULT)

        # remove last fully connected layer
        self.feature_extractor = nn.Sequential(*list(resnet_model.children())[:-1])

        # define GRU
        self.gru = nn.GRU(input_size=512, hidden_size=128, batch_first=True) # hidden_size möglicherweise anpassen

        # final classifier
        self.fc = nn.Linear(128, num_classes) 


    def forward(self,x):
        batch_size, seq_len, channels, height, width = x.size()

        # use ResNet for each picture in the sequence
        x = x.view(batch_size * seq_len, channels, height, width)
        with torch.no_grad(): # ResNet eingefroren
            features = self.feature_extractor(x)

        # bring features into right sequential form
        features = torch.flatten(features, 1)
        features = features.view(batch_size, seq_len, -1)

        gru_out, hn = self.gru(features)

        last_hidden_state = hn[-1]

        output = self.fc(last_hidden_state)
        return output


model = Experiment3(num_classes=4)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001) # SGD durch Adam ersetzt, da es anscheinend für GRUs stabiler und schneller sein soll

In [ ]:
# load dataset
class VideoDataset(Dataset):
    def __init__(self, metadata_csv, transform=None):
        self.df = pd.read_csv(metadata_csv)
        self.transform = transform
        self.seq_len = 16 # vielleicht nochmal anpassen

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_folder = row['frame_dir'] # schauen, ob das wirklich die richtigen codes sind
        video_class = int(row['class_id']) # schauen, ob das wirklich die richtigen codes sind

        img_sequence = sorted(os.listdir(video_folder))


# define transformations for our dataset
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

### Training Loop (muss noch angepasst werden)

In [ ]:
# training loop 
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        # Print the results for the current epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')

In [ ]:
# train model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model_three_quarter = resnet_model.to(device)

train(model_three_quarter, train_loader_three_quarter, val_loader_three_quarter, criterion, optimizer, num_epochs=10)

# save checkpoint of model
checkpoint = {
    "model_three_quarter_state_dict": model_three_quarter.state_dict(),
    "optimizer_model_three_quarter_state_dict": optimizer.state_dict(),
    "epoch": 10,
}

torch.save(
    checkpoint,
    "model_three_quarter_checkpoint.pth"
)